# Lenguajes y Autómatas II (SCD-1016)
## Sesión 09 · Reporte Formal de Errores Semánticos y Diagnóstico

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Qué debe decir un compilador cuando el programa está mal?
>
> **Objetivo de la sesión:** Emitir diagnósticos con archivo, línea, columna y nivel, en el formato que las herramientas saben leer, y seguir analizando después del primer error.

**Entregable:** tu formateador en artesano_sesion09_c.c, el harness en verde y la bitácora

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Implementas reportar(...), que arma archivo:línea:columna: error: mensaje dentro del buffer que recibe.

Al final de la sesión tendrás: **Un formateador de diagnósticos que produce mensajes idénticos al estilo de GCC y no desborda el buffer.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 09](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion09.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion09.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion09.h"

void reportar(const char *archivo, int linea, int columna, const char *mensaje, char *salida, int cap);
```

- El formato es exacto: sin espacios extra ni punto final.
- Nunca escribas más de cap-1 caracteres: el buffer es del llamador.
- El resultado siempre termina en '\0'.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion09_c.c` junto con `artesano_sesion09_c.c`
> (o `ia_sesion09_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `main.c:14:5` | `main.c:14:5: error: tipo incompatible` |
| 2 | `a.c:1:1` | `a.c:1:1: error: x` |
| 3 | `linea 0` | `x.c:0:0: error: y` |
| 4 | `cap=10 trunca` | `main.c:14` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 09 - Diagnostico con linea y columna ===
Caso                               esperado:              obtenido:              resultado
[ 1] main.c:14:5
     esperado: main.c:14:5: error: tipo incompatible
     obtenido: main.c:14:5: error: tipo incompatible
     -> OK
[ 2] a.c:1:1
     esperado: a.c:1:1: error: x
     obtenido: a.c:1:1: error: x
     -> OK
[ 3] linea 0
     esperado: x.c:0:0: error: y
     obtenido: x.c:0:0: error: y
     -> OK
[ 4] cap=10 trunca                      esperado: main.c:14    obtenido: main.c:14    OK

RESULTADO: 4/4 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion09.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El formato coincide carácter por carácter con el estilo de GCC.
- [ ] El buffer nunca se desborda y siempre queda terminado.
- [ ] Los cuatro casos del harness salen OK con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion09_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion09.h
/* api_sesion09.h - contrato de la Sesion 09
 * Diagnóstico con línea y columna
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION09_H
#define API_SESION09_H

void reportar(const char *archivo, int linea, int columna, const char *mensaje, char *salida, int cap);

#endif /* API_SESION09_H */


In [ ]:
%%writefile artesano_sesion09_c.c
/* artesano_sesion09_c.c - Fase 1: El Artesano (sin IA)
 * Diagnóstico con línea y columna.
 *
 * Implementa lo que declara api_sesion09.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion09.h"
#include <stdio.h>
#include <string.h>

void reportar(const char *archivo, int linea, int columna, const char *mensaje, char *salida, int cap) {
    (void)archivo; (void)linea; (void)columna; (void)mensaje; (void)cap;
    /* TODO: arma el diagnostico con snprintf y respeta cap */
    if (salida != NULL && cap > 0) salida[0] = '\0';
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion09_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion09_c.c
/* ia_sesion09_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion09.h. No puede traer main().
 */
#include "api_sesion09.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion09_c.c
/* harness_sesion09_c.c - auditoria automatica de la Sesion 09
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion09_c.c artesano_sesion09_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion09.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 09 - Diagnostico con linea y columna ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char b[128]; reportar("main.c", 14, 5, "tipo incompatible", b, 128); verificar_str_largo("main.c:14:5", "main.c:14:5: error: tipo incompatible", b); }
    { char b[64]; reportar("a.c", 1, 1, "x", b, 64); verificar_str_largo("a.c:1:1", "a.c:1:1: error: x", b); }
    { char b[64]; reportar("x.c", 0, 0, "y", b, 64); verificar_str_largo("linea 0", "x.c:0:0: error: y", b); }
    { char b[10]; reportar("main.c", 14, 5, "tipo incompatible", b, 10); verificar_str("cap=10 trunca", "main.c:14", b); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion09.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion09_c.c artesano_sesion09_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion09_c.c ia_sesion09_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 4/4 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion09.md` a `auditorias/auditoria_sesion09.md` en tu fork.
2. Adjunta la salida del harness (las 4 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
